In [1]:
from chapters.chapter_5_agents_intro import agent_executor
from chats import llm

/Users/ikartavost-pc/pet_projects/ai/langchain-course/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [3]:
from langchain_core.tools import  tool

@tool
def add(x: float, y: float) -> float:
    """Add 'x' and 'y'."""
    return x + y

@tool
def multiply(x: float, y: float) -> float:
    """Multiply 'x' and 'y'."""
    return x * y

@tool
def exponentiate(x: float, y: float) -> float:
    """Raise 'x' to the power of 'y'."""
    return x ** y

@tool
def subtract(x: float, y: float) -> float:
    """Subtract 'x' from 'y'."""
    return y - x

In [4]:
import json

llm_output_string = "{\"x\": 5, \"y\": 2}"
llm_output_dict = json.loads(llm_output_string)
llm_output_dict

{'x': 5, 'y': 2}

In [5]:
exponentiate.func(**llm_output_dict)

25

In [6]:
from langchain_core.prompts import  ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "You're a helpful assistant. When answering a user's question "
        "you should first use one of the tools provided. After using a "
        "tool the tool output will be provided in the "
        "'scratchpad' below. If you have an answer in the "
        "scratchpad you should not use any more tools and "
        "instead answer directly to the user."
    )),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad"),
])

In [7]:
from langchain_core.runnables.base import  RunnableSerializable

tools = [add, subtract, multiply, exponentiate]

agent: RunnableSerializable = (
    {
        "input": lambda x: x["input"],
        "chat_history": lambda x: x["chat_history"],
        "agent_scratchpad": lambda x:x.get("agent_scratchpad",[])
    }
    | prompt
    | llm.bind_tools(tools, tool_choice="any")
)

In [8]:
tool_call = agent.invoke({"input": "What is 10 + 10", "chat_history": []})
tool_call

AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:8b', 'created_at': '2026-03-05T20:42:02.24409Z', 'done': True, 'done_reason': 'stop', 'total_duration': 21830796917, 'load_duration': 3681265084, 'prompt_eval_count': 354, 'prompt_eval_duration': 3929314167, 'eval_count': 144, 'eval_duration': 13344668207, 'logprobs': None, 'model_name': 'qwen3:8b', 'model_provider': 'ollama'}, id='lc_run--019cbfbc-153b-7bd3-bc09-737f5b6bff56-0', tool_calls=[{'name': 'add', 'args': {'x': 10, 'y': 10}, 'id': '6ab308f6-3343-4f4b-af67-27fd878af2d7', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 354, 'output_tokens': 144, 'total_tokens': 498})

In [9]:
tool_call.tool_calls

[{'name': 'add',
  'args': {'x': 10, 'y': 10},
  'id': '6ab308f6-3343-4f4b-af67-27fd878af2d7',
  'type': 'tool_call'}]

In [10]:
name2tool = {tool.name: tool.func for tool in tools}

In [11]:
name2tool

{'add': <function __main__.add(x: float, y: float) -> float>,
 'subtract': <function __main__.subtract(x: float, y: float) -> float>,
 'multiply': <function __main__.multiply(x: float, y: float) -> float>,
 'exponentiate': <function __main__.exponentiate(x: float, y: float) -> float>}

In [14]:
tool_exec_content = name2tool[tool_call.tool_calls[0]["name"]](**tool_call.tool_calls[0]["args"])
tool_exec_content

20

In [15]:
from langchain_core.messages import ToolMessage

tool_exec = ToolMessage(
    content=f"The {tool_call.tool_calls[0]['name']} tool returned {tool_exec_content}",
    tool_call_id =tool_call.tool_calls[0]['id']
)

out = agent.invoke({
    "input": "What is 10 + 10",
    "chat_history": [],
    "agent_scratchpad": [tool_call, tool_exec]
})
out

AIMessage(content='The result of 10 + 10 is **20**.', additional_kwargs={}, response_metadata={'model': 'qwen3:8b', 'created_at': '2026-03-05T20:50:32.799709Z', 'done': True, 'done_reason': 'stop', 'total_duration': 20247824041, 'load_duration': 3679941541, 'prompt_eval_count': 398, 'prompt_eval_duration': 4467854542, 'eval_count': 124, 'eval_duration': 11363829708, 'logprobs': None, 'model_name': 'qwen3:8b', 'model_provider': 'ollama'}, id='lc_run--019cbfc3-e5c5-7931-a228-8a8e31d0bb06-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 398, 'output_tokens': 124, 'total_tokens': 522})

In [17]:
agent: RunnableSerializable = (
    {
        "input": lambda x: x["input"],
        "chat_history": lambda x: x["chat_history"],
        "agent_scratchpad": lambda x:x.get("agent_scratchpad",[])
    }
    | prompt
    | llm.bind_tools(tools, tool_choice="auto")
)

In [18]:
tool_call = agent.invoke({"input": "What is 10 + 10", "chat_history": []})
tool_call

AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:8b', 'created_at': '2026-03-05T20:54:16.135763Z', 'done': True, 'done_reason': 'stop', 'total_duration': 16271459292, 'load_duration': 110185208, 'prompt_eval_count': 354, 'prompt_eval_duration': 1866081583, 'eval_count': 144, 'eval_duration': 13594425372, 'logprobs': None, 'model_name': 'qwen3:8b', 'model_provider': 'ollama'}, id='lc_run--019cbfc7-5db5-7661-960a-8a2c6de5e013-0', tool_calls=[{'name': 'add', 'args': {'x': 10, 'y': 10}, 'id': '8c774fc0-7fa8-4430-b33c-6723048f1d9e', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 354, 'output_tokens': 144, 'total_tokens': 498})

In [20]:
tool_exec_content = name2tool[tool_call.tool_calls[0]["name"]](**tool_call.tool_calls[0]["args"])
tool_exec = ToolMessage(
    content=f"The {tool_call.tool_calls[0]['name']} tool returned {tool_exec_content}",
    tool_call_id =tool_call.tool_calls[0]['id']
)

out = agent.invoke({
    "input": "What is 10 + 10",
    "chat_history": [],
    "agent_scratchpad": [tool_call, tool_exec]
})
out

AIMessage(content='The result of 10 + 10 is **20**.', additional_kwargs={}, response_metadata={'model': 'qwen3:8b', 'created_at': '2026-03-05T20:55:52.8649Z', 'done': True, 'done_reason': 'stop', 'total_duration': 14635979000, 'load_duration': 116048000, 'prompt_eval_count': 398, 'prompt_eval_duration': 2154791792, 'eval_count': 124, 'eval_duration': 11696892666, 'logprobs': None, 'model_name': 'qwen3:8b', 'model_provider': 'ollama'}, id='lc_run--019cbfc8-ddf1-7823-a4e9-cf9866929a02-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 398, 'output_tokens': 124, 'total_tokens': 522})

In [21]:
@tool
def final_answer(answer: str, tools_used: list[str]) -> str:
    """Use this tool to provide a final answer to the user.
    The answer should be in natural language as this will be provided
    to the user directly. The tools_used must include a list of tool
    names that were used within the `scratchpad`.
    """
    return {"answer": answer, "tools_used": tools_used}

In [27]:
tools = [final_answer, add, subtract, multiply, exponentiate]

In [31]:
name2tool = {tool.name: tool.func for tool in tools}

agent: RunnableSerializable = (
    {
        "input": lambda x: x["input"],
        "chat_history": lambda x: x["chat_history"],
        "agent_scratchpad": lambda x: x.get("agent_scratchpad", [])
    }
    | prompt
    | llm.bind_tools(tools, tool_choice="any")  # we're forcing tool use again
)

In [32]:
tool_call = agent.invoke({"input": "What is 10 + 10", "chat_history": []})
tool_call.tool_calls

[{'name': 'add',
  'args': {'y': 10, 'x': 10},
  'id': 'c0c91256-1dcf-4fbb-8118-261ce61f6719',
  'type': 'tool_call'}]

In [33]:
tool_out = name2tool[tool_call.tool_calls[0]["name"]](
    **tool_call.tool_calls[0]["args"]
)

tool_exec = ToolMessage(
    content=f"The {tool_call.tool_calls[0]['name']} tool returned {tool_out}",
    tool_call_id=tool_call.tool_calls[0]["id"]
)

out = agent.invoke({
    "input": "What is 10 + 10",
    "chat_history": [],
    "agent_scratchpad": [tool_call, tool_exec]
})
out

AIMessage(content='The result of 10 + 10 is 20.', additional_kwargs={}, response_metadata={'model': 'qwen3:8b', 'created_at': '2026-03-05T21:04:01.44341Z', 'done': True, 'done_reason': 'stop', 'total_duration': 16551044291, 'load_duration': 133343083, 'prompt_eval_count': 499, 'prompt_eval_duration': 669829500, 'eval_count': 152, 'eval_duration': 15010533669, 'logprobs': None, 'model_name': 'qwen3:8b', 'model_provider': 'ollama'}, id='lc_run--019cbfd0-4afa-78f2-a579-785caedabc17-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 499, 'output_tokens': 152, 'total_tokens': 651})

In [34]:
out.tool_calls

[]

In [35]:
from  langchain_core.messages import  BaseMessage, HumanMessage, AIMessage

class CustomAgentExecutor:
    chat_history: list[BaseMessage]

    def __init__(self, max_iterations: int = 3):
        self.chat_history = []
        self.max_iterations = max_iterations
        self.agent: RunnableSerializable = (
            {
                "input": lambda x: x["input"],
                "chat_history": lambda x: x["chat_history"],
                "agent_scratchpad": lambda x: x.get("agent_scratchpad", [])
            }
            | prompt
            | llm.bind_tools(tools, tool_choice="any")  # we're forcing tool use again
        )

    def invoke(self, input: str) -> dict:
        # invoke the agent but we do this iteratively in a loop until
        # reaching a final answer
        count = 0
        agent_scratchpad = []
        while count < self.max_iterations:
            # invoke a step for the agent to generate a tool call
            tool_call = self.agent.invoke({
                "input": input,
                "chat_history": self.chat_history,
                "agent_scratchpad": agent_scratchpad
            })
            # add initial tool call to scratchpad
            agent_scratchpad.append(tool_call)
            # otherwise we execute the tool and add it's output to the agent scratchpad
            tool_name = tool_call.tool_calls[0]["name"]
            tool_args = tool_call.tool_calls[0]["args"]
            tool_call_id = tool_call.tool_calls[0]["id"]
            tool_out = name2tool[tool_name](**tool_args)
            # add the tool output to the agent scratchpad
            tool_exec = ToolMessage(
                content=f"{tool_out}",
                tool_call_id=tool_call_id
            )
            agent_scratchpad.append(tool_exec)
            # add a print so we can see intermediate steps
            print(f"{count}: {tool_name}({tool_args})")
            count += 1
            # if the tool call is the final answer tool, we stop
            if tool_name == "final_answer":
                break
        # add the final output to the chat history
        final_answer = tool_out["answer"]
        self.chat_history.extend([
            HumanMessage(content=input),
            AIMessage(content=final_answer)
        ])
        # return the final answer in dict form
        return json.dumps(tool_out)

In [36]:
agent_executor = CustomAgentExecutor()

In [ ]:
agent_executor.invoke("What is 10 * 7.4 / 5")

0: multiply({'x': 10, 'y': 7.4})
